# 03a · Build the action-clip dataset (Steps 1–3)

**Owner:** Ben · **Goal:** turn the Volleyball Dataset (Ibrahim et al., CVPR 2016) into small, balanced, ready-to-train arrays of player clips.

**What you'll have at the end**

| Output | Where | What it is |
|---|---|---|
| `train_000.npz`, `val_000.npz`, `test_000.npz`, … | `WORK/action_clips/` | Arrays of clips, each **16 frames × 112 × 112 × 3** (uint8, RGB), plus integer labels |
| `labels.csv` | same folder | One row per clip: split, shard, position, video, frame, label, original action, box |
| `class_weights.json` | same folder | Loss weights for the 5 classes, from the train split |
| `checks/*.png` | `WORK/checks/` | The visual checks below; commit these to git, not the data |

**Classes:** `spike`, `set`, `dig`, `block`, `other` (other = waiting, moving, standing, jumping, falling).

Run the cells top to bottom. Every step ends with a **check** cell; don't move on until its output looks right.

## Step 1 · Setup

### 1.1 Pick a runtime

- **Colab:** Runtime → Change runtime type → **T4 GPU**. This notebook doesn't need the GPU (it only reads images and crops them), but the next one does, and you can keep the same session.
- **Kaggle:** Settings → Accelerator → GPU T4 ×2, and turn **Internet on**. Kaggle sessions last longer and have more disk, which helps here.

The raw dataset is large (55 videos × 41 frames per labelled moment), so **download it to the runtime's own disk, not into Google Drive**. A free Drive has 15 GB, and reading thousands of small files from Drive is very slow. Only the finished clip arrays (a few GB) go to Drive at the end.

### 1.2 Install and import

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "tqdm", "gdown"], check=False)

import json, math, random, shutil, time
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 0
random.seed(SEED); np.random.seed(SEED)
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| OpenCV", cv2.__version__)

### 1.3 Paths and settings

`VB_ROOT` is the folder that holds the video folders `0` … `54`. `WORK` is fast local disk for the output. Change these if your layout differs; everything else reads from here.

In [ ]:
DEFAULT_ROOT = "/content/volleyball" if IN_COLAB else "/kaggle/input/volleyball" if IN_KAGGLE else "./volleyball"
DEFAULT_WORK = "/content/work" if IN_COLAB else "/kaggle/working" if IN_KAGGLE else "./work"
VB_ROOT = Path(os.environ.get("VB_ROOT", DEFAULT_ROOT))
WORK = Path(os.environ.get("WORK_DIR", DEFAULT_WORK))
CLIPS_DIR, CHECKS_DIR = WORK / "action_clips", WORK / "checks"
DRIVE_DIR = Path("/content/drive/MyDrive/volleyball/action/data")   # Colab only: where the finished clips are copied

CFG = dict(
    before=8, after=7,          # frames before / after the labelled frame -> 16 frames
    out=112,                    # crop size in pixels: small enough to keep the clip set manageable; VGG16 accepts it
    scale=1.3,                  # crop side = 1.3 x the larger box side, keeps arms and ball in view
    shard_size=int(os.environ.get("SHARD_SIZE", 500)),   # clips per .npz file (~300 MB in memory)
    other_cap_ratio=1.0,        # 'other' clips per split = this x the largest real class in that split
)
CLASSES = ["spike", "set", "dig", "block", "other"]
ACTION_MAP = {"spiking": "spike", "setting": "set", "digging": "dig", "blocking": "block",
              "waiting": "other", "moving": "other", "standing": "other", "jumping": "other", "falling": "other"}
TRAIN = [1,3,6,7,10,13,15,16,18,22,23,31,32,36,38,39,40,41,42,48,50,52,53,54]
VAL   = [0,2,8,12,17,19,24,26,27,28,30,33,46,49,51]
TEST  = [4,5,9,11,14,20,21,25,29,34,35,37,43,44,45,47]
SPLIT_OF = {**{v: "train" for v in TRAIN}, **{v: "val" for v in VAL}, **{v: "test" for v in TEST}}

for d in (WORK, CLIPS_DIR, CHECKS_DIR): d.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(WORK).free / 1e9
print(f"VB_ROOT={VB_ROOT}\nWORK={WORK}\nFree disk at WORK: {free_gb:.0f} GB")

## Step 2 · Get and parse the dataset

### 2.1 Extract only the frames we need

The dataset folder (from the **"NEW Download Link"** on <https://github.com/mostafa-saad/deep-activity-rec>) holds:

| File | Size | Use |
|---|---|---|
| `volleyball_.zip` | 57.76 GB | **The dataset.** 41 frames around every labelled moment |
| `videos_sample.zip` | 52.6 MB | A small sample: handy for a first test run |
| `volleyball_tracking_annotation.zip` | 12.7 MB | Player boxes tracked across all frames (optional, later) |
| `Facing Download issues` | doc | The authors' tips if Google blocks the download; read it if you hit a quota error |

Unzipping all 57.76 GB would fill most of a Colab disk and take hours. We only need **16 of the 41 frames** (−8 … +7) plus the `annotations.txt` files, so this cell reads the zip's index, works out exactly which files are needed, and extracts only those, about 40% of the data. It reads them in the order they sit in the zip (much faster through Drive), and it's **resumable**: files already extracted are skipped, so after a disconnect just run it again.

**Setup:**
1. In the shared Drive folder, right-click `volleyball_.zip` → *Organize → Add shortcut* → *My Drive*. (Optionally also `videos_sample.zip`.)
2. Run the cell once with `ZIP_IN_DRIVE` pointing at `videos_sample.zip` to test the whole notebook in a few minutes, if that zip has the same layout. Then switch to `volleyball_.zip` for the real run.
3. Before the real run, check free disk with `!df -h /content`. You need roughly 25 GB free.

Skip this cell if `VB_ROOT` already holds folders `0`…`54`.

In [ ]:
import zipfile
ZIP_IN_DRIVE = Path(os.environ.get("ZIP_PATH", "/content/drive/MyDrive/volleyball_.zip"))   # <- your shortcut's path
EXTRACT_DIR = Path(os.environ.get("EXTRACT_DIR", "/content/vb_raw" if IN_COLAB else "./vb_raw"))
SELECTIVE = True        # False = extract everything (needs ~60 GB free)

def have_dataset(root):
    return root.exists() and any((p / "annotations.txt").exists() for p in root.iterdir() if p.is_dir())

def extract_needed(zip_path, dest, before=CFG["before"], after=CFG["after"], selective=SELECTIVE):
    zf = zipfile.ZipFile(zip_path)
    infos = {i.filename: i for i in zf.infolist()}
    anns = sorted(n for n in infos if n.endswith("annotations.txt"))
    if not anns:
        raise RuntimeError("No annotations.txt inside this zip. Is it the right file?")
    need = set(anns)
    for a in anns:
        prefix = a[: -len("annotations.txt")]             # e.g. 'volleyball_/videos/12/'
        for line in zf.read(a).decode().splitlines():
            tok = line.split()
            if not tok: continue
            fid = int(tok[0].split(".")[0])
            ks = range(fid - before, fid + after + 1) if selective else range(fid - 20, fid + 21)
            need.update(f"{prefix}{fid}/{k}.jpg" for k in ks)
    members = sorted((infos[n] for n in need if n in infos), key=lambda i: i.header_offset)
    todo = [i for i in members if not (dest / i.filename).exists()]
    gb = sum(i.file_size for i in todo) / 1e9
    print(f"{len(anns)} videos in zip | {len(members):,} files needed | {len(todo):,} still to extract ({gb:.1f} GB)")
    missing = len(need) - len(members)
    if missing: print(f"note: {missing:,} expected frame files are not in the zip (edge frames); the clip loader handles that")
    for info in tqdm(todo, desc="extracting", unit="file", mininterval=5):
        zf.extract(info, dest)
    return dest / Path(anns[0]).parent.parent           # folder that holds 0..54

if not have_dataset(VB_ROOT):
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    if not ZIP_IN_DRIVE.exists():
        raise FileNotFoundError(f"{ZIP_IN_DRIVE} not found. Check the shortcut name with !ls /content/drive/MyDrive/")
    t0 = time.time()
    VB_ROOT = extract_needed(ZIP_IN_DRIVE, EXTRACT_DIR)
    print(f"Done in {(time.time() - t0) / 60:.1f} min")
print("Dataset present:", have_dataset(VB_ROOT), "at", VB_ROOT)

### 2.2 Check the folder structure

Expected: one folder per video (`0` … `54`), each with `annotations.txt` and one sub-folder per labelled frame holding the images named by frame number: **16** (frames −8 … +7) if you extracted selectively in 2.1, otherwise 41.

In [ ]:
def check_dataset(root):
    vids = sorted(int(p.name) for p in root.iterdir() if p.is_dir() and p.name.isdigit())
    missing = sorted(set(SPLIT_OF) - set(vids))
    rows = []
    for v in vids:
        clips = [p for p in (root / str(v)).iterdir() if p.is_dir()]
        sample = clips[0] if clips else None
        n_img = len(list(sample.glob("*.jpg"))) if sample else 0
        img = cv2.imread(str(next(sample.glob("*.jpg")))) if n_img else None
        rows.append(dict(video=v, split=SPLIT_OF.get(v, "?"), clips=len(clips),
                         imgs_in_first_clip=n_img,
                         size=f"{img.shape[1]}x{img.shape[0]}" if img is not None else "?",
                         has_annotations=(root / str(v) / "annotations.txt").exists()))
    info = pd.DataFrame(rows)
    print(f"{len(vids)} video folders found; missing from the official split: {missing or 'none'}")
    print("Videos not at 1280x720:", info.loc[info["size"] != "1280x720", ["video", "size"]].values.tolist())
    expected = CFG["before"] + CFG["after"] + 1 if SELECTIVE else 41
    print(f"Clip folders with fewer than {expected} images in the sampled clip:", info.loc[info.imgs_in_first_clip < expected, "video"].tolist())
    return info

ds_info = check_dataset(VB_ROOT)
ds_info.head()

**Check:** 55 videos, none missing, 16 images per clip (41 if not selective), and 8 videos at 1920×1080 (2, 37, 38, 39, 40, 41, 44, 45). A video with fewer images is usually a broken download; re-extract that folder.

### 2.3 Parse `annotations.txt`

Each line: frame ID (`29885.jpg`), the group activity (`r_spike`), then **5 tokens per player**: a box (X Y W H) and an action word. Sources disagree on whether the action comes first or last, so the parser finds the word wherever it is.

In [ ]:
def parse_annotations(root):
    rows = []
    vids = sorted(int(p.name) for p in root.iterdir() if p.is_dir() and p.name.isdigit())
    for vid in vids:
        if vid not in SPLIT_OF:
            continue
        for line_no, line in enumerate(open(root / str(vid) / "annotations.txt")):
            tok = line.split()
            if len(tok) < 7:
                continue
            fid, activity, rest = int(tok[0].split(".")[0]), tok[1], tok[2:]
            if len(rest) % 5:
                print(f"warning: video {vid} line {line_no}: {len(rest)} player tokens, not a multiple of 5")
            for i in range(0, len(rest) - len(rest) % 5, 5):
                g = rest[i:i + 5]
                words = [t for t in g if not t.lstrip("-").isdigit()]
                nums = [int(t) for t in g if t.lstrip("-").isdigit()]
                if len(words) != 1 or len(nums) != 4 or words[0] not in ACTION_MAP:
                    print(f"warning: video {vid} frame {fid}: could not read player tokens {g}")
                    continue
                x, y, w, h = nums
                rows.append(dict(vid=vid, fid=fid, activity=activity, x=x, y=y, w=w, h=h,
                                 raw=words[0], label=ACTION_MAP[words[0]], split=SPLIT_OF[vid]))
    return pd.DataFrame(rows)

ann = parse_annotations(VB_ROOT)
ann.to_csv(WORK / "annotations_parsed.csv", index=False)
print(f"{len(ann):,} player annotations in {ann.groupby(['vid', 'fid']).ngroups:,} labelled frames\n")
print("Original actions:\n", ann.raw.value_counts().to_string(), "\n")
print("Merged classes by split:")
ann.groupby(["split", "label"]).size().unstack(fill_value=0)[CLASSES]

**Check:** about 4,830 labelled frames in total. In the published counts, standing is by far the largest (≈38,700), and spiking, setting, digging and blocking are each roughly 1,200–2,500. If you see far fewer rows, some lines failed to parse; read the warnings above.

### 2.4 Look at the boxes

Draw the boxes and actions on three random labelled frames. **This is the most important check in the notebook.** If boxes sit beside the players instead of on them, X Y isn't the top-left corner (or W H aren't width and height). Fix that in the parser before building anything.

In [ ]:
def frame_path(root, vid, fid, k):
    return root / str(vid) / str(fid) / f"{k}.jpg"

def draw_check(root, ann, n=3, seed=SEED):
    keys = ann[["vid", "fid"]].drop_duplicates().sample(n, random_state=seed).values
    fig, axes = plt.subplots(n, 1, figsize=(12, 5.5 * n))
    for ax, (vid, fid) in zip(np.atleast_1d(axes), keys):
        img = cv2.cvtColor(cv2.imread(str(frame_path(root, vid, fid, fid))), cv2.COLOR_BGR2RGB)
        for r in ann[(ann.vid == vid) & (ann.fid == fid)].itertuples():
            cv2.rectangle(img, (r.x, r.y), (r.x + r.w, r.y + r.h), (255, 220, 0), 2)
            cv2.putText(img, r.raw, (r.x, max(12, r.y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 220, 0), 2)
        ax.imshow(img); ax.set_title(f"video {vid}, frame {fid}"); ax.axis("off")
    fig.tight_layout(); fig.savefig(CHECKS_DIR / "boxes_check.png", dpi=80); plt.show()

draw_check(VB_ROOT, ann)

## Step 3 · Build the player-clip dataset

### 3.1 How a clip is made

For each labelled player: take frames **−8 … +7** around the labelled frame (16 frames), and crop the **same square** from each:

- centre = the box centre; side = 1.3 × the larger of box width and height (keeps raised arms and the ball);
- pad with black where the square runs off the image;
- resize to 112 × 112 and convert BGR → RGB.

The box comes from the labelled frame only; players move a little in half a second, which is fine. **The same `square_crop` must be used on our own videos later** (Step 8), so it lives in its own cell and will be copied into `pipeline.ipynb`.

In [ ]:
def square_crop(img, x, y, w, h, scale=CFG["scale"], out=CFG["out"]):
    # Square crop centred on box (x, y, w, h), padded with black at the borders, resized to out x out.
    cx, cy, s = x + w / 2, y + h / 2, max(w, h) * scale
    s = max(int(round(s)), 2)
    x0, y0 = int(round(cx - s / 2)), int(round(cy - s / 2))
    pad = s
    big = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)
    crop = big[y0 + pad:y0 + pad + s, x0 + pad:x0 + pad + s]
    return cv2.resize(crop, (out, out), interpolation=cv2.INTER_AREA)

def read_window(root, vid, fid, before=CFG["before"], after=CFG["after"]):
    # Read the 16 frames around a labelled frame once; a missing frame is replaced by its nearest neighbour.
    frames, last = [], None
    for k in range(fid - before, fid + after + 1):
        img = cv2.imread(str(frame_path(root, vid, fid, k)))
        if img is None:
            img = last if last is not None else cv2.imread(str(frame_path(root, vid, fid, fid)))
        frames.append(img); last = img
    return frames

def clip_from_window(frames, x, y, w, h):
    return np.stack([cv2.cvtColor(square_crop(f, x, y, w, h), cv2.COLOR_BGR2RGB) for f in frames])

# quick self-test on one annotation
r0 = ann.iloc[0]
c0 = clip_from_window(read_window(VB_ROOT, r0.vid, r0.fid), r0.x, r0.y, r0.w, r0.h)
print("clip shape:", c0.shape, c0.dtype)   # expect (16, 112, 112, 3) uint8

### 3.2 Balance the classes

"Other" is roughly 10× bigger than all four real actions together. If we kept it all, a model could score high accuracy by always answering "other". So, **separately in each split**, keep every spike, set, dig and block, and randomly sample "other" down to the size of the largest real class. Validation and test are balanced the same way, so their scores are comparable to training.

Class weights (inverse frequency in the **train** split, normalised to average 1) go into the loss in Step 4 to even out what's left.

In [ ]:
def balance(ann, ratio=CFG["other_cap_ratio"], seed=SEED):
    parts = []
    for split, g in ann.groupby("split"):
        real = g[g.label != "other"]
        cap = int(real.label.value_counts().max() * ratio) if len(real) else 0
        other = g[g.label == "other"]
        parts += [real, other.sample(min(cap, len(other)), random_state=seed)]
    return pd.concat(parts).sort_values(["split", "vid", "fid"]).reset_index(drop=True)

sel = balance(ann)
counts = sel.groupby(["split", "label"]).size().unstack(fill_value=0)[CLASSES]
train_counts = counts.loc["train"].astype(float)
weights = (1 / train_counts.clip(lower=1)); weights = weights / weights.mean()
class_weights = {c: round(float(weights[c]), 4) for c in CLASSES}
json.dump(class_weights, open(CLIPS_DIR / "class_weights.json", "w"), indent=2)
print("Clips to build:", len(sel)); print("Class weights:", class_weights)
counts

### 3.3 Build and save the clips

Clips are written in **shards** of `shard_size` clips per compressed `.npz` file, with `X` = clips (N × 16 × 112 × 112 × 3, uint8) and `y` = class indices. Frames are read once per labelled moment and reused for every player in it.

The cell is **resumable**: a finished split leaves a `*.done` marker and is skipped on re-run, so a dropped session only costs the split in progress. Expect this to take a while (it's reading and cropping tens of thousands of images); the progress bar shows the rate.

In [ ]:
def build_split(split, sel, root=VB_ROOT, out_dir=CLIPS_DIR, shard_size=CFG["shard_size"]):
    done_flag = out_dir / f"{split}.done"
    if done_flag.exists():
        print(f"{split}: already built, skipping"); return pd.read_csv(out_dir / f"labels_{split}.csv")
    g = sel[sel.split == split]
    X, y, meta, shard, rows = [], [], [], 0, []
    def flush():
        nonlocal X, y, meta, shard
        if not X: return
        np.savez_compressed(out_dir / f"{split}_{shard:03d}.npz", X=np.stack(X), y=np.array(y, np.int8))
        for i, m in enumerate(meta): rows.append({**m, "shard": shard, "idx": i})
        X, y, meta, shard = [], [], [], shard + 1
    for (vid, fid), players in tqdm(g.groupby(["vid", "fid"]), desc=split):
        frames = read_window(root, vid, fid)
        for r in players.itertuples():
            X.append(clip_from_window(frames, r.x, r.y, r.w, r.h)); y.append(CLASSES.index(r.label))
            meta.append(dict(split=split, vid=vid, fid=fid, label=r.label, raw=r.raw, x=r.x, y=r.y, w=r.w, h=r.h))
            if len(X) >= shard_size: flush()
    flush()
    labels = pd.DataFrame(rows); labels.to_csv(out_dir / f"labels_{split}.csv", index=False)
    done_flag.touch(); return labels

t0 = time.time()
labels = pd.concat([build_split(s, sel) for s in ["train", "val", "test"]], ignore_index=True)
labels.to_csv(CLIPS_DIR / "labels.csv", index=False)
size_gb = sum(p.stat().st_size for p in CLIPS_DIR.glob("*.npz")) / 1e9
print(f"Built {len(labels):,} clips in {time.time() - t0:.0f} s; {size_gb:.2f} GB on disk")

### 3.4 Check the clips

Three checks, all saved to `checks/` for your report:

1. Shapes, dtype and counts per split match the table in 3.2.
2. A grid of 20 random clips (middle frame) with labels: spikers and blockers should be in the air at the net, diggers low, setters with hands up.
3. All 16 frames of one spike: you should see the movement, and the player should stay roughly inside the crop.

In [ ]:
def load_shard(split, shard):
    z = np.load(CLIPS_DIR / f"{split}_{shard:03d}.npz"); return z["X"], z["y"]

X0, y0 = load_shard("train", 0)
print("first train shard:", X0.shape, X0.dtype, "labels:", np.bincount(y0, minlength=5))
assert X0.shape[1:] == (16, CFG["out"], CFG["out"], 3) and X0.dtype == np.uint8
print(labels.groupby(["split", "label"]).size().unstack(fill_value=0)[CLASSES])

# 20 random clips, middle frame
pick = labels.sample(min(20, len(labels)), random_state=SEED)
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
cache = {}
for ax, r in zip(axes.ravel(), pick.itertuples()):
    key = (r.split, r.shard)
    if key not in cache: cache[key] = load_shard(*key)[0]
    ax.imshow(cache[key][r.idx][CFG["before"]]); ax.set_title(f"{r.label} ({r.raw})\nv{r.vid} f{r.fid}", fontsize=9); ax.axis("off")
fig.tight_layout(); fig.savefig(CHECKS_DIR / "clip_grid.png", dpi=80); plt.show()

# one spike, all 16 frames
sp = labels[labels.label == "spike"].iloc[0]
clip = load_shard(sp.split, sp.shard)[0][sp.idx]
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(clip[i]); ax.set_title(f"t={i - CFG['before']:+d}", fontsize=9); ax.axis("off")
fig.suptitle(f"spike · video {sp.vid} frame {sp.fid}"); fig.tight_layout()
fig.savefig(CHECKS_DIR / "clip_strip_spike.png", dpi=80); plt.show()

big_vids = ds_info.loc[ds_info["size"] != "1280x720", "video"].tolist()
print("Clips from 1920x1080 videos:", int(labels.vid.isin(big_vids).sum()), "(they should look the same size as the rest in the grid)")

### 3.5 Save to Drive (Colab) or keep as Kaggle output

The training notebook reads from `CLIPS_DIR`. On Colab, local disk is wiped when the session ends, so copy the finished folder to Drive once (a few GB). On Kaggle, *Save Version* keeps `/kaggle/working` as an output you can attach to the training notebook.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    shutil.copytree(CLIPS_DIR, DRIVE_DIR / "action_clips", dirs_exist_ok=True)
    shutil.copytree(CHECKS_DIR, DRIVE_DIR / "checks", dirs_exist_ok=True)
    print(f"Copied to {DRIVE_DIR} in {time.time() - t0:.0f} s")
else:
    print("Not on Colab: clips stay in", CLIPS_DIR)

## Done checklist (Steps 1–3)

- [ ] 2.2: 55 videos, 16 images per clip, 8 at 1920×1080
- [ ] 2.3: about 4,830 labelled frames; counts pasted into your notes
- [ ] 2.4: boxes sit on the players
- [ ] 3.4: shapes correct; the clip grid and the spike strip look right
- [ ] `action_clips/` copied to Drive (or saved as Kaggle output)
- [ ] Commit this notebook (outputs cleared by `nbstripout`) and the `checks/*.png` images; **never** commit the `.npz` files

Next: `03_actions.ipynb`, Step 4 (single-frame baseline).